# Ethos-U55 mapping notebook

minimal end-to-end run for loading the architecture, generating mappings, and benchmarking the final yolo-world mapping

## step 1 - load ethos_u55 arch with chosen hyperparams

In [1]:
from pathlib import Path

import accelforge as af
from accelforge.mapper import Metrics

from milestone_1.load_ethos_u55 import build_ethos_u55_jinja_data, load_ethos_u55_spec

ARCH_FILE = "arch/ethos_u55.yaml"
WORKLOAD_MATMUL = "workload/matmul.yaml"
WORKLOAD_YOLO = "workload/yolo_world.yaml"
MAPPINGS_DIR = Path("mappings")
MAPPINGS_DIR.mkdir(exist_ok=True)

ARCH_OVERRIDES = {
    "num_macs": 128,
    "system_preset": "high_end_embedded",
    "memory_mode": "dedicated_384kb",
    "clock_hz": 1.0e9,
    "M": 1024,
    "N": 512,
    "K": 256,
}

jinja_data = build_ethos_u55_jinja_data(**ARCH_OVERRIDES)
print("loaded ethos_u55 architecture with hyperparams")
for key in ["NUM_MACS", "CLOCK_HZ", "SYSTEM_SRAM_SIZE_BITS", "LOCAL_BUFFER_SIZE_BITS"]:
    print(f"{key}: {jinja_data[key]}")

loaded ethos_u55 architecture with hyperparams
NUM_MACS: 128
CLOCK_HZ: 1000000000.0
SYSTEM_SRAM_SIZE_BITS: 3145728
LOCAL_BUFFER_SIZE_BITS: 262144


## setup helpers for mapping + saving outputs

In [2]:
def run_mapper(spec: af.Spec):
    spec.mapper.metrics = Metrics.LATENCY | Metrics.ENERGY
    try:
        return spec.map_workload_to_arch(
            print_progress=False,
            print_number_of_pmappings=False,
        )
    except TypeError:
        return spec.map_workload_to_arch()


# accelforge uses flattened column names like Total<SEP>energy (see lab notebooks)
_AF_SEP = "<SEP>"


def _total_metric_column(df, tail: str):
    """pick Total aggregate column, not per-component EinsumZ<SEP>energy<SEP>..."""
    if df is None:
        return None
    exact = f"Total{_AF_SEP}{tail}"
    for c in df.columns:
        if str(c) == exact:
            return c
    for c in df.columns:
        s = str(c)
        if s.startswith("Total") and s.endswith(tail) and _AF_SEP in s:
            return c
    return None


def min_edp_index(df):
    energy_col = _total_metric_column(df, "energy")
    latency_col = _total_metric_column(df, "latency")
    if energy_col and latency_col:
        return (df[energy_col].astype(float) * df[latency_col].astype(float)).idxmin()
    return None


def select_best_mapping(mappings):
    data = getattr(mappings, "data", None)
    idx = min_edp_index(data)
    if idx is not None:
        return mappings[idx]

    # fallback path when mappings.data does not expose energy/latency columns
    best_obj = None
    best_edp = None
    for i in range(len(mappings)):
        candidate = mappings[i]
        try:
            edp = float(candidate.energy()) * float(candidate.latency())
        except Exception:
            continue
        if best_edp is None or edp < best_edp:
            best_edp = edp
            best_obj = candidate

    if best_obj is None:
        available_cols = list(data.columns) if data is not None else []
        raise RuntimeError(
            "mapper returned no selectable result with energy/latency metrics "
            f"available columns: {available_cols}"
        )

    return best_obj


def save_mapping_yaml(mapping_obj, output_path: Path):
    # lab pattern: best_row.mapping().to_yaml() (see lab_4 lab.ipynb)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    inner = getattr(mapping_obj, "mapping", None)
    if callable(inner):
        try:
            inner = inner()
        except Exception:
            inner = None
    targets = [t for t in (inner, mapping_obj) if t is not None]

    for target in targets:
        for method_name in ["to_yaml", "save_yaml", "dump_yaml", "save", "to_file"]:
            method = getattr(target, method_name, None)
            if not callable(method):
                continue
            try:
                result = method(str(output_path))
                if output_path.exists():
                    return output_path
                if isinstance(result, str) and result.strip():
                    output_path.write_text(result)
                    return output_path
            except TypeError:
                try:
                    result = method()
                    if isinstance(result, str) and result.strip():
                        output_path.write_text(result)
                        return output_path
                except Exception:
                    pass
            except Exception:
                pass

    raise RuntimeError("could not save mapping yaml from mapper output")


def first_layer_workload_and_renames(batch_size: int = 1):
    jinja = {"BATCH_SIZE": batch_size}
    full_workload = af.Workload.from_yaml(WORKLOAD_YOLO, top_key="workload", jinja_parse_data=jinja)
    renames = af.Renames.from_yaml(WORKLOAD_YOLO, top_key="renames", jinja_parse_data=jinja)
    first_workload = af.Workload(
        einsums=[full_workload.einsums[0]],
        rank_sizes=full_workload.rank_sizes,
        bits_per_value=full_workload.bits_per_value,
    )
    return first_workload, renames

## step 2 - run mapper on simple architecture (matmul.yaml)

In [3]:
print("Running `matmul` mapping...\n")
spec_matmul = load_ethos_u55_spec(
    workload_yaml=WORKLOAD_MATMUL,
    arch_yaml=ARCH_FILE,
    **ARCH_OVERRIDES,
)
matmul_mappings = run_mapper(spec_matmul)
# print("mapper columns:", list(getattr(matmul_mappings, "data", []).columns))
matmul_best = select_best_mapping(matmul_mappings)

matmul_mapping_path = MAPPINGS_DIR / "matmul_mapping.yaml"
save_mapping_yaml(matmul_best, matmul_mapping_path)

print("saved matmul mapping:", matmul_mapping_path)
print("energy:", float(matmul_best.energy()))
print("latency:", float(matmul_best.latency()))

saved matmul mapping: mappings/matmul_mapping.yaml
energy: 0.0024640193532695265
latency: 0.008388607762753963


## step 3 - run mapper on first yolo world layer

In [ ]:
print("Running `yolo-world-first` mapping...\n")
spec_yolo_first = load_ethos_u55_spec(
    workload_yaml=WORKLOAD_YOLO,
    workload_jinja_parse={"BATCH_SIZE": 1},
    arch_yaml=ARCH_FILE,
    **ARCH_OVERRIDES,
)
first_workload, yolo_renames = first_layer_workload_and_renames(batch_size=1)
spec_yolo_first.workload = first_workload
spec_yolo_first.renames = yolo_renames

yolo_first_mappings = run_mapper(spec_yolo_first)
yolo_first_best = select_best_mapping(yolo_first_mappings)

yolo_first_mapping_path = MAPPINGS_DIR / "yolo_world_first_layer_mapping.yaml"
save_mapping_yaml(yolo_first_best, yolo_first_mapping_path)

print("saved first-layer yolo mapping:", yolo_first_mapping_path)
print("energy:", float(yolo_first_best.energy()))
print("latency:", float(yolo_first_best.latency()))

Running `yolo-world-first` mapping...



## step 4 - run mapper on full yolo world and save to mappings/

In [ ]:
print("Running `yolo-world-all` mapping...\n")
spec_yolo_full = load_ethos_u55_spec(
    workload_yaml=WORKLOAD_YOLO,
    workload_jinja_parse={"BATCH_SIZE": 1},
    arch_yaml=ARCH_FILE,
    **ARCH_OVERRIDES,
)
yolo_full_mappings = run_mapper(spec_yolo_full)
yolo_full_best = select_best_mapping(yolo_full_mappings)

yolo_full_mapping_path = MAPPINGS_DIR / "yolo_world_full_mapping.yaml"
save_mapping_yaml(yolo_full_best, yolo_full_mapping_path)

print("saved full yolo mapping:", yolo_full_mapping_path)
print("energy:", float(yolo_full_best.energy()))
print("latency:", float(yolo_full_best.latency()))

## step 5 - benchmark final mapping (energy, size, latency)

In [ ]:
spec_benchmark = load_ethos_u55_spec(
    workload_yaml=WORKLOAD_YOLO,
    mapping_yaml=str(yolo_full_mapping_path),
    workload_jinja_parse={"BATCH_SIZE": 1},
    arch_yaml=ARCH_FILE,
    **ARCH_OVERRIDES,
)
benchmark_result = spec_benchmark.evaluate_mapping()

energy_j = float(benchmark_result.energy())
latency_cycles = float(benchmark_result.latency())
mapping_size_bytes = yolo_full_mapping_path.stat().st_size

benchmark = {
    "energy_j": energy_j,
    "latency_cycles": latency_cycles,
    "mapping_size_bytes": mapping_size_bytes,
}

print("full yolo benchmark")
for key, value in benchmark.items():
    print(f"{key}: {value}")

benchmark